# LSTM

## Import libraries

In [1]:
import os
import sys
import pandas as pd
import lightning as L
import torch.nn as nn
import torch
from torch.optim import Adam
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from tsfresh.utilities.dataframe_functions import (
    roll_time_series,
)
from tsfresh import extract_features, select_features
from tsfresh.utilities.dataframe_functions import impute
from tsfresh.feature_extraction import ComprehensiveFCParameters
import matplotlib.pylab as plt
from sklearn.preprocessing import MinMaxScaler
from lightning.pytorch.callbacks import EarlyStopping

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from logger.logger import Logger
from utils.constants import FEATURE_SELECTION_LOG_FILE_BASE
from tabular_database_driver.postgre_sql_driver import PostgreSQLDriver
from dtos.tabular_database_driver_dtos.postgre_sql_connection_dto import (
    PostgreSQLConnectionDto,
)

## Parameters

In [2]:
TARGET_COLUMN = "close"
TARGET_COLUMN

'close'

In [3]:
# Inclusive
TRAIN_RANGE = (2000, 2021)
VALIDATION_RANGE = (2022, 2023)
TEST_RANGE = (2024, 2025)

In [4]:
INPUT_SIZE = None
OUTPUT_SIZE = 1

## Hyper Parameters

In [5]:
MAX_EPOCHS = 100
LEARNING_RATE = 0.001
PATIENCE = 30

## Start

### Get data

In [6]:
my_logger = Logger(file_name=f"{FEATURE_SELECTION_LOG_FILE_BASE}/vn_index/test")

In [7]:
my_connection_model = PostgreSQLConnectionDto(
    logger=my_logger,
    host=os.getenv("POSTGRES_HOST"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    port=os.getenv("POSTGRES_PORT"),
    database=os.getenv("GOLD_POSTGRES_DATABASE"),
)

In [8]:
my_postgresql_driver = PostgreSQLDriver(logger=my_logger)
my_postgresql_driver.connect(my_connection_model)

<DatabaseExecutionStatus.SUCCESS: 'success'>

In [9]:
vn_index_df = my_postgresql_driver.select(
    schema_name="stock_market", table_name="vn_index"
)

In [10]:
dtype_map = {
    "date": str,
    "open": float,
    "high": float,
    "low": float,
    "close": float,
    "volume": float,
}

vn_index_df = (
    vn_index_df.astype(dtype_map)
    .dropna(subset=["close"])
    .sort_values(by=["date"])
    .reset_index(drop=True)
)

In [11]:
vn_index_df

,date,open,high,low,close,volume
0,2000-07-28,100.000000,100.000000,100.000000,100.000000,4200.0
1,2000-07-29,100.516667,100.516667,100.516667,100.516667,6233.0
2,2000-07-30,101.033333,101.033333,101.033333,101.033333,8267.0
3,2000-07-31,101.550000,101.550000,101.550000,101.550000,10300.0
4,2000-08-01,102.465000,102.465000,102.465000,102.465000,5300.0
...,...,...,...,...,...,...
9099,2025-06-26,1368.730000,1370.610000,1360.780000,1365.670000,598388700.0
9100,2025-06-27,1369.130000,1373.280000,1362.090000,1371.440000,677408300.0
9101,2025-06-28,1371.340000,1374.620000,1365.403333,1372.983333,663601667.0
9102,2025-06-29,1373.550000,1375.960000,1368.716667,1374.526667,649795033.0


In [12]:
vn_index_df_close = vn_index_df[["date", "close"]]
vn_index_df_close

,date,close
0,2000-07-28,100.000000
1,2000-07-29,100.516667
2,2000-07-30,101.033333
3,2000-07-31,101.550000
4,2000-08-01,102.465000
...,...,...
9099,2025-06-26,1365.670000
9100,2025-06-27,1371.440000
9101,2025-06-28,1372.983333
9102,2025-06-29,1374.526667


In [13]:
vn_index_df_close.loc[:, "stock"] = "vn_index"
vn_index_df_close

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_32352\2662759176.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  vn_index_df_close.loc[:, "stock"] = "vn_index"


,date,close,stock
0,2000-07-28,100.000000,vn_index
1,2000-07-29,100.516667,vn_index
2,2000-07-30,101.033333,vn_index
3,2000-07-31,101.550000,vn_index
4,2000-08-01,102.465000,vn_index
...,...,...,...
9099,2025-06-26,1365.670000,vn_index
9100,2025-06-27,1371.440000,vn_index
9101,2025-06-28,1372.983333,vn_index
9102,2025-06-29,1374.526667,vn_index


In [14]:
vn_index_df_close_rolled = roll_time_series(
    vn_index_df_close,
    column_id="stock",
    column_sort="date",
    max_timeshift=20,
    min_timeshift=5,
)
vn_index_df_close_rolled

Rolling: 100%|██████████| 50/50 [00:06<00:00,  7.47it/s]


,date,close,stock,id
0,2000-07-28,100.000000,vn_index,"(vn_index, 2000-08-02)"
1,2000-07-29,100.516667,vn_index,"(vn_index, 2000-08-02)"
2,2000-07-30,101.033333,vn_index,"(vn_index, 2000-08-02)"
3,2000-07-31,101.550000,vn_index,"(vn_index, 2000-08-02)"
4,2000-08-01,102.465000,vn_index,"(vn_index, 2000-08-02)"
...,...,...,...,...
187111,2025-06-26,1365.670000,vn_index,"(vn_index, 2025-06-30)"
187112,2025-06-27,1371.440000,vn_index,"(vn_index, 2025-06-30)"
187113,2025-06-28,1372.983333,vn_index,"(vn_index, 2025-06-30)"
187114,2025-06-29,1374.526667,vn_index,"(vn_index, 2025-06-30)"


In [15]:
X = extract_features(
    vn_index_df_close_rolled.drop("stock", axis=1),
    column_id="id",
    column_sort="date",
    column_value="close",
    impute_function=impute,
    default_fc_parameters=ComprehensiveFCParameters(),
)

Feature Extraction: 100%|██████████| 50/50 [01:23<00:00,  1.67s/it]


In [16]:
X

close__variance_larger_than_standard_deviation  \
vn_index 2000-08-02                                             1.0   
         2000-08-03                                             1.0   
         2000-08-04                                             1.0   
         2000-08-05                                             1.0   
         2000-08-06                                             1.0   
...                                                             ...   
         2025-06-26                                             1.0   
         2025-06-27                                             1.0   
         2025-06-28                                             1.0   
         2025-06-29                                             1.0   
         2025-06-30                                             1.0   

                     close__has_duplicate_max  close__has_duplicate_min  \
vn_index 2000-08-02                       0.0                       0.0   
         2000-08-03                       0.0                       0.0   
         2000-08-04                       0.0                       0.0   
         2000-08-05                       0.0                       0.0   
         2000-08-06                       0.0                       0.0   
...                                       ...                       ...   
         2025-06-26                       0.0                       0.0   
         2025-06-27                       0.0                       0.0   
         2025-06-28                       0.0                       0.0   
         2025-06-29                       0.0                       0.0   
         2025-06-30                       0.0                       0.0   

                     close__has_duplicate  close__sum_values  \
vn_index 2000-08-02                   0.0         608.945000   
         2000-08-03                   0.0         713.235000   
         2000-08-04                   0.0         818.435000   
         2000-08-05                   0.0         924.208333   
         2000-08-06                   0.0        1030.555000   
...                                   ...                ...   
         2025-06-26                   0.0       28103.350000   
         2025-06-27                   0.0       28144.900000   
         2025-06-28                   0.0       28194.433333   
         2025-06-29                   0.0       28251.950000   
         2025-06-30                   0.0       28317.450000   

                     close__abs_energy  close__mean_abs_change  \
vn_index 2000-08-02       6.181024e+04                0.676000   
         2000-08-03       7.268664e+04                0.715000   
         2000-08-04       8.375368e+04                0.742857   
         2000-08-05       9.494168e+04                0.721667   
         2000-08-06       1.062513e+05                0.705185   
...                                ...                     ...   
         2025-06-26       3.761674e+07                5.039000   
         2025-06-27       3.772898e+07                5.005500   
         2025-06-28       3.786255e+07                4.760667   
         2025-06-29       3.801735e+07                4.515833   
         2025-06-30       3.819333e+07                4.310000   

                     close__mean_change  \
vn_index 2000-08-02            0.676000   
         2000-08-03            0.715000   
         2000-08-04            0.742857   
         2000-08-05            0.721667   
         2000-08-06            0.705185   
...                                 ...   
         2025-06-26            1.789000   
         2025-06-27            2.399500   
         2025-06-28            2.798667   
         2025-06-29            3.197833   
         2025-06-30            2.992000   

                     close__mean_second_derivative_central  close__median  \
vn_index 2000-08-02                               0.049792     101.291667   
         2000-08-03               

In [17]:
X = X.set_index(X.index.map(lambda x: x[1]), drop=True)
X.index.name = "last_date"
X

,close__variance_larger_than_standard_deviation,close__has_duplicate_max,close__has_duplicate_min,close__has_duplicate,close__sum_values,close__abs_energy,close__mean_abs_change,close__mean_change,close__mean_second_derivative_central,close__median,...,close__fourier_entropy__bins_5,close__fourier_entropy__bins_10,close__fourier_entropy__bins_100,close__permutation_entropy__dimension_3__tau_1,close__permutation_entropy__dimension_4__tau_1,close__permutation_entropy__dimension_5__tau_1,close__permutation_entropy__dimension_6__tau_1,close__permutation_entropy__dimension_7__tau_1,close__query_similarity_count__query_None__threshold_0.0,close__mean_n_absolute_max__number_of_maxima_7
last_date,,,,,,,,,,,,,,,,,,,,,
2000-08-02,1.0,0.0,0.0,0.0,608.945000,6.181024e+04,0.676000,0.676000,0.049792,101.291667,...,0.562335,0.562335,1.386294,-0.000000,-0.000000,-0.000000,-0.000000,2.615631,0.0,572.868571
2000-08-03,1.0,0.0,0.0,0.0,713.235000,7.268664e+04,0.715000,0.715000,0.039333,101.550000,...,0.562335,0.562335,1.386294,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,572.868571
2000-08-04,1.0,0.0,0.0,0.0,818.435000,8.375368e+04,0.742857,0.742857,0.032778,102.007500,...,0.500402,0.500402,1.332179,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,102.633571
2000-08-05,1.0,0.0,0.0,0.0,924.208333,9.494168e+04,0.721667,0.721667,0.004048,102.465000,...,0.500402,0.500402,1.332179,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,103.384524
2000-08-06,1.0,0.0,0.0,0.0,1030.555000,1.062513e+05,0.705185,0.705185,0.003542,102.922500,...,0.450561,0.450561,1.242453,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,104.143571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-26,1.0,0.0,0.0,0.0,28103.350000,3.761674e+07,5.039000,1.789000,0.141053,1338.110000,...,0.304636,0.600166,1.294545,1.489767,2.120208,2.507026,2.512659,2.523211,0.0,1359.562857
2025-06-27,1.0,0.0,0.0,0.0,28144.900000,3.772898e+07,5.005500,2.399500,0.321316,1346.830000,...,0.304636,0.600166,1.159589,1.458585,2.120208,2.507026,2.512659,2.523211,0.0,1362.334286
2025-06-28,1.0,0.0,0.0,0.0,28194.433333,3.786255e+07,4.760667,2.798667,0.210088,1347.690000,...,0.304636,0.304636,1.294545,1.380452,2.014123,2.507026,2.512659,2.523211,0.0,1365.290000


In [18]:
y = vn_index_df_close.set_index("date").sort_index()["close"].shift(-1)
y = pd.DataFrame(y)
y

,close
date,
2000-07-28,100.516667
2000-07-29,101.033333
2000-07-30,101.550000
2000-07-31,102.465000
2000-08-01,103.380000
...,...
2025-06-26,1371.440000
2025-06-27,1372.983333
2025-06-28,1374.526667


In [19]:
y = y[y.index.isin(X.index)]
X = X[X.index.isin(y.index)]

In [20]:
X

,close__variance_larger_than_standard_deviation,close__has_duplicate_max,close__has_duplicate_min,close__has_duplicate,close__sum_values,close__abs_energy,close__mean_abs_change,close__mean_change,close__mean_second_derivative_central,close__median,...,close__fourier_entropy__bins_5,close__fourier_entropy__bins_10,close__fourier_entropy__bins_100,close__permutation_entropy__dimension_3__tau_1,close__permutation_entropy__dimension_4__tau_1,close__permutation_entropy__dimension_5__tau_1,close__permutation_entropy__dimension_6__tau_1,close__permutation_entropy__dimension_7__tau_1,close__query_similarity_count__query_None__threshold_0.0,close__mean_n_absolute_max__number_of_maxima_7
last_date,,,,,,,,,,,,,,,,,,,,,
2000-08-02,1.0,0.0,0.0,0.0,608.945000,6.181024e+04,0.676000,0.676000,0.049792,101.291667,...,0.562335,0.562335,1.386294,-0.000000,-0.000000,-0.000000,-0.000000,2.615631,0.0,572.868571
2000-08-03,1.0,0.0,0.0,0.0,713.235000,7.268664e+04,0.715000,0.715000,0.039333,101.550000,...,0.562335,0.562335,1.386294,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,572.868571
2000-08-04,1.0,0.0,0.0,0.0,818.435000,8.375368e+04,0.742857,0.742857,0.032778,102.007500,...,0.500402,0.500402,1.332179,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,102.633571
2000-08-05,1.0,0.0,0.0,0.0,924.208333,9.494168e+04,0.721667,0.721667,0.004048,102.465000,...,0.500402,0.500402,1.332179,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,103.384524
2000-08-06,1.0,0.0,0.0,0.0,1030.555000,1.062513e+05,0.705185,0.705185,0.003542,102.922500,...,0.450561,0.450561,1.242453,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,104.143571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-26,1.0,0.0,0.0,0.0,28103.350000,3.761674e+07,5.039000,1.789000,0.141053,1338.110000,...,0.304636,0.600166,1.294545,1.489767,2.120208,2.507026,2.512659,2.523211,0.0,1359.562857
2025-06-27,1.0,0.0,0.0,0.0,28144.900000,3.772898e+07,5.005500,2.399500,0.321316,1346.830000,...,0.304636,0.600166,1.159589,1.458585,2.120208,2.507026,2.512659,2.523211,0.0,1362.334286
2025-06-28,1.0,0.0,0.0,0.0,28194.433333,3.786255e+07,4.760667,2.798667,0.210088,1347.690000,...,0.304636,0.304636,1.294545,1.380452,2.014123,2.507026,2.512659,2.523211,0.0,1365.290000


In [21]:
y

,close
date,
2000-08-02,104.290000
2000-08-03,105.200000
2000-08-04,105.773333
2000-08-05,106.346667
2000-08-06,106.920000
...,...
2025-06-26,1371.440000
2025-06-27,1372.983333
2025-06-28,1374.526667


### Create train validation test

In [22]:
TRAIN_RANGE[0], TRAIN_RANGE[1]

(2000, 2021)

In [23]:
X_train = X[f"{TRAIN_RANGE[0]}" :f"{TRAIN_RANGE[1] + 1}"]
X_train

,close__variance_larger_than_standard_deviation,close__has_duplicate_max,close__has_duplicate_min,close__has_duplicate,close__sum_values,close__abs_energy,close__mean_abs_change,close__mean_change,close__mean_second_derivative_central,close__median,...,close__fourier_entropy__bins_5,close__fourier_entropy__bins_10,close__fourier_entropy__bins_100,close__permutation_entropy__dimension_3__tau_1,close__permutation_entropy__dimension_4__tau_1,close__permutation_entropy__dimension_5__tau_1,close__permutation_entropy__dimension_6__tau_1,close__permutation_entropy__dimension_7__tau_1,close__query_similarity_count__query_None__threshold_0.0,close__mean_n_absolute_max__number_of_maxima_7
last_date,,,,,,,,,,,,,,,,,,,,,
2000-08-02,1.0,0.0,0.0,0.0,608.945000,6.181024e+04,0.676000,0.676000,0.049792,101.291667,...,0.562335,0.562335,1.386294,-0.000000,-0.000000,-0.000000,-0.000000,2.615631,0.0,572.868571
2000-08-03,1.0,0.0,0.0,0.0,713.235000,7.268664e+04,0.715000,0.715000,0.039333,101.550000,...,0.562335,0.562335,1.386294,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,572.868571
2000-08-04,1.0,0.0,0.0,0.0,818.435000,8.375368e+04,0.742857,0.742857,0.032778,102.007500,...,0.500402,0.500402,1.332179,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,102.633571
2000-08-05,1.0,0.0,0.0,0.0,924.208333,9.494168e+04,0.721667,0.721667,0.004048,102.465000,...,0.500402,0.500402,1.332179,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,103.384524
2000-08-06,1.0,0.0,0.0,0.0,1030.555000,1.062513e+05,0.705185,0.705185,0.003542,102.922500,...,0.450561,0.450561,1.242453,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,104.143571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-12-27,1.0,0.0,0.0,0.0,30934.680000,4.557145e+07,5.044500,2.105500,-0.056579,1476.610000,...,1.033562,1.540306,2.397895,1.497896,2.370135,2.670120,2.772589,2.708050,0.0,1481.491429
2021-12-28,1.0,0.0,0.0,0.0,30982.300000,4.571150e+07,5.015000,2.076000,-0.252632,1477.030000,...,1.366711,1.767761,2.397895,1.497896,2.351257,2.588573,2.772589,2.708050,0.0,1483.811429
2021-12-29,1.0,0.0,0.0,0.0,31015.250000,4.580833e+07,4.688000,0.892000,-0.108684,1477.330000,...,1.414279,1.641735,2.098274,1.616282,2.351257,2.588573,2.772589,2.708050,0.0,1484.822857


In [24]:
y_train = y[f"{TRAIN_RANGE[0]}" :f"{TRAIN_RANGE[1] + 1}"]
y_train

,close
date,
2000-08-02,104.290000
2000-08-03,105.200000
2000-08-04,105.773333
2000-08-05,106.346667
2000-08-06,106.920000
...,...
2021-12-27,1494.390000
2021-12-28,1485.820000
2021-12-29,1485.970000


In [25]:
X_train_selected = select_features(X_train, y_train[TARGET_COLUMN])
X_train_selected

,close__sum_values,close__maximum,close__absolute_maximum,close__minimum,close__benford_correlation,"close__agg_linear_trend__attr_""slope""__chunk_len_10__f_agg_""var""","close__agg_linear_trend__attr_""stderr""__chunk_len_5__f_agg_""max""",close__mean_n_absolute_max__number_of_maxima_7,close__c3__lag_2,"close__agg_linear_trend__attr_""stderr""__chunk_len_10__f_agg_""var""",...,close__ratio_beyond_r_sigma__r_2.5,"close__fft_coefficient__attr_""imag""__coeff_10","close__cwt_coefficients__coeff_9__w_2__widths_(2, 5, 10, 20)",close__longest_strike_above_mean,close__large_standard_deviation__r_0.30000000000000004,"close__cwt_coefficients__coeff_11__w_2__widths_(2, 5, 10, 20)","close__cwt_coefficients__coeff_10__w_2__widths_(2, 5, 10, 20)",close__index_mass_quantile__q_0.6,close__autocorrelation__lag_9,close__ratio_beyond_r_sigma__r_0.5
last_date,,,,,,,,,,,,,,,,,,,,,
2000-08-02,608.945000,103.380000,103.380000,100.00,0.864123,-16.978800,0.000000,572.868571,1.045243e+06,13.818124,...,0.000000,0.172690,0.262493,3.0,1.0,0.191928,0.190537,0.666667,-0.333385,0.666667
2000-08-03,713.235000,104.290000,104.290000,100.00,0.864123,-16.978800,0.000000,572.868571,1.056712e+06,13.818124,...,0.000000,0.172690,0.262493,3.0,1.0,0.191928,0.190537,0.714286,-0.333385,0.714286
2000-08-04,818.435000,105.200000,105.200000,100.00,0.864123,-16.978800,0.000000,102.633571,1.068638e+06,13.818124,...,0.000000,0.172690,0.262493,4.0,1.0,0.191928,0.190537,0.625000,-0.333385,0.750000
2000-08-05,924.208333,105.773333,105.773333,100.00,0.864123,-16.978800,0.000000,103.384524,1.080970e+06,13.818124,...,0.000000,0.172690,0.262493,4.0,1.0,0.191928,0.190537,0.666667,-0.333385,0.777778
2000-08-06,1030.555000,106.346667,106.346667,100.00,0.864123,-16.978800,0.000000,104.143571,1.093572e+06,13.818124,...,0.000000,0.172690,59.574627,5.0,1.0,0.191928,0.190537,0.700000,-2.142677,0.800000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-12-27,30934.680000,1488.880000,1488.880000,1446.77,0.864123,-48.766084,0.493938,1481.491429,3.201745e+09,0.531717,...,0.047619,-5.673272,4.021763,10.0,0.0,5.476924,4.131697,0.619048,-0.126890,0.523810
2021-12-28,30982.300000,1494.390000,1494.390000,1452.87,0.864123,-29.297492,0.845903,1483.811429,3.211178e+09,18.936114,...,0.000000,12.624671,4.414730,10.0,0.0,6.639488,5.475928,0.619048,-0.041900,0.428571
2021-12-29,31015.250000,1494.390000,1494.390000,1456.96,0.864123,-13.084172,1.806106,1484.822857,3.219784e+09,42.443335,...,0.000000,-14.383113,5.760181,6.0,0.0,6.223545,6.638799,0.619048,0.051626,0.428571


In [26]:
INPUT_SIZE = X_train_selected.shape[1]
INPUT_SIZE

315

In [27]:
VALIDATION_RANGE[0], VALIDATION_RANGE[1]

(2022, 2023)

In [28]:
X_validation_selected = X[f"{VALIDATION_RANGE[0]}" :f"{VALIDATION_RANGE[1] + 1}"][
    X_train_selected.columns
]
X_validation_selected

,close__sum_values,close__maximum,close__absolute_maximum,close__minimum,close__benford_correlation,"close__agg_linear_trend__attr_""slope""__chunk_len_10__f_agg_""var""","close__agg_linear_trend__attr_""stderr""__chunk_len_5__f_agg_""max""",close__mean_n_absolute_max__number_of_maxima_7,close__c3__lag_2,"close__agg_linear_trend__attr_""stderr""__chunk_len_10__f_agg_""var""",...,close__ratio_beyond_r_sigma__r_2.5,"close__fft_coefficient__attr_""imag""__coeff_10","close__cwt_coefficients__coeff_9__w_2__widths_(2, 5, 10, 20)",close__longest_strike_above_mean,close__large_standard_deviation__r_0.30000000000000004,"close__cwt_coefficients__coeff_11__w_2__widths_(2, 5, 10, 20)","close__cwt_coefficients__coeff_10__w_2__widths_(2, 5, 10, 20)",close__index_mass_quantile__q_0.6,close__autocorrelation__lag_9,close__ratio_beyond_r_sigma__r_0.5
last_date,,,,,,,,,,,,,,,,,,,,,
2022-01-01,31105.321667,1505.1050,1505.1050,1456.96,0.864123,-2.259802,1.010796,1491.910714,3.243119e+09,65.922666,...,0.000000,20.741410,1.929978,7.0,0.0,-18.113164,-8.172057,0.619048,-0.473750,0.428571
2022-01-02,31145.265000,1511.9300,1511.9300,1456.96,0.864123,-0.903474,1.588364,1495.767857,3.252391e+09,88.642803,...,0.047619,-18.037320,-7.884076,8.0,0.0,-19.756803,-18.114000,0.619048,-0.417663,0.571429
2022-01-03,31187.810000,1518.7550,1518.7550,1456.96,0.864123,-20.060112,1.802351,1500.472857,3.262600e+09,51.317218,...,0.000000,21.271310,-17.825196,8.0,0.0,-10.945980,-19.757693,0.619048,-0.332533,0.619048
2022-01-04,31237.370000,1525.5800,1525.5800,1456.96,0.864123,-20.151062,2.343493,1506.131429,3.276275e+09,72.823999,...,0.000000,-16.643599,-19.468939,5.0,0.0,1.351916,-10.947016,0.619048,-0.336745,0.666667
2022-01-05,31284.370000,1525.5800,1525.5800,1456.96,0.864123,-21.406530,2.998018,1510.934286,3.291147e+09,100.660857,...,0.000000,18.649084,-10.658360,6.0,0.0,8.080746,1.350933,0.619048,-0.313792,0.714286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-27,23346.860000,1127.6300,1127.6300,1091.88,0.864123,-49.363779,3.609295,1124.535714,1.365683e+09,23.250665,...,0.000000,-4.632530,-14.222252,7.0,1.0,-20.002139,-19.221444,0.619048,-0.998157,0.809524
2023-12-28,23354.300000,1128.9300,1128.9300,1091.88,0.864123,-69.533880,3.869525,1125.527143,1.365100e+09,18.675034,...,0.000000,4.686420,-19.001995,6.0,1.0,-16.728193,-20.002294,0.619048,-1.003888,0.761905
2023-12-29,23359.790000,1129.9300,1129.9300,1091.88,0.864123,-85.957144,3.745480,1126.311429,1.364530e+09,12.818803,...,0.000000,-3.817382,-19.782265,6.0,1.0,-11.476449,-16.728308,0.619048,-0.942133,0.761905


In [29]:
y_validation = y[f"{VALIDATION_RANGE[0]}" :f"{VALIDATION_RANGE[1] + 1}"]
y_validation

,close
date,
2022-01-01,1511.9300
2022-01-02,1518.7550
2022-01-03,1525.5800
2022-01-04,1522.5000
2022-01-05,1528.5700
...,...
2023-12-27,1128.9300
2023-12-28,1129.9300
2023-12-29,1130.3775


In [30]:
TEST_RANGE[0], TEST_RANGE[1]

(2024, 2025)

In [31]:
X_test_selected = X[f"{TEST_RANGE[0]}" :f"{TEST_RANGE[1] + 1}"][
    X_train_selected.columns
]
X_test_selected

,close__sum_values,close__maximum,close__absolute_maximum,close__minimum,close__benford_correlation,"close__agg_linear_trend__attr_""slope""__chunk_len_10__f_agg_""var""","close__agg_linear_trend__attr_""stderr""__chunk_len_5__f_agg_""max""",close__mean_n_absolute_max__number_of_maxima_7,close__c3__lag_2,"close__agg_linear_trend__attr_""stderr""__chunk_len_10__f_agg_""var""",...,close__ratio_beyond_r_sigma__r_2.5,"close__fft_coefficient__attr_""imag""__coeff_10","close__cwt_coefficients__coeff_9__w_2__widths_(2, 5, 10, 20)",close__longest_strike_above_mean,close__large_standard_deviation__r_0.30000000000000004,"close__cwt_coefficients__coeff_11__w_2__widths_(2, 5, 10, 20)","close__cwt_coefficients__coeff_10__w_2__widths_(2, 5, 10, 20)",close__index_mass_quantile__q_0.6,close__autocorrelation__lag_9,close__ratio_beyond_r_sigma__r_0.5
last_date,,,,,,,,,,,,,,,,,,,,,
2024-01-01,23376.825000,1131.272500,1131.272500,1091.88,0.864123,-51.177766,3.861833,1128.745000,1.370886e+09,22.361403,...,0.0,2.394830,-7.278382,8.0,1.0,-4.804178,-5.897500,0.619048,-0.716439,0.761905
2024-01-02,23380.915000,1131.720000,1131.720000,1091.88,0.864123,-20.237993,2.508342,1129.329286,1.375035e+09,24.526860,...,0.0,-1.482129,-5.676844,9.0,1.0,-2.333476,-4.804264,0.619048,-0.621100,0.761905
2024-01-03,23410.885000,1144.170000,1144.170000,1091.88,0.864123,-14.056878,1.786385,1132.460714,1.381966e+09,15.012784,...,0.0,5.003116,-4.586280,10.0,0.0,1.101303,-2.334103,0.619048,-0.454517,0.714286
2024-01-04,23451.475000,1150.720000,1150.720000,1091.88,0.864123,-16.835223,0.986104,1135.573571,1.390245e+09,17.531755,...,0.0,-2.362717,-2.116934,11.0,0.0,3.967352,1.100454,0.619048,-0.307096,0.809524
2024-01-05,23503.855000,1154.680000,1154.680000,1091.88,0.864123,-29.198487,1.378546,1139.109286,1.400298e+09,24.230132,...,0.0,7.476372,1.316072,11.0,0.0,5.732087,3.966256,0.619048,-0.244252,0.809524
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-26,28103.350000,1366.770000,1366.770000,1310.57,0.864123,-19.652801,2.210113,1359.562857,2.390856e+09,29.770882,...,0.0,5.324101,-8.683325,10.0,1.0,9.408850,0.527119,0.619048,-0.431397,0.809524
2025-06-27,28144.900000,1371.440000,1371.440000,1310.57,0.864123,-30.458812,1.126588,1362.334286,2.405000e+09,14.580064,...,0.0,-0.099843,0.787291,11.0,1.0,12.840584,9.407981,0.619048,-0.350009,0.761905
2025-06-28,28194.433333,1372.983333,1372.983333,1310.57,0.864123,-62.196244,2.132187,1365.290000,2.421563e+09,2.554403,...,0.0,2.255915,9.666879,12.0,1.0,9.857666,12.839548,0.619048,-0.276512,0.714286


In [32]:
y_test = y[f"{TEST_RANGE[0]}" :f"{TEST_RANGE[1] + 1}"]
y_test

,close
date,
2024-01-01,1131.720000
2024-01-02,1144.170000
2024-01-03,1150.720000
2024-01-04,1154.680000
2024-01-05,1156.516667
...,...
2025-06-26,1371.440000
2025-06-27,1372.983333
2025-06-28,1374.526667


### Create scaler

In [33]:
X_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X_train_scaler = X_scaler.fit_transform(X_train_selected)
y_train_scaler = y_scaler.fit_transform(y_train)

X_validation_scaler = X_scaler.transform(X_validation_selected)
y_validation_scaler = y_scaler.transform(y_validation)

X_test_scaler = X_scaler.transform(X_test_selected)
y_test_scaler = y_scaler.transform(y_test)

In [34]:
X_train_scaler.shape

(7822, 315)

In [35]:
y_train_scaler.shape

(7822, 1)

In [36]:
X_validation_scaler.shape

(730, 315)

In [37]:
y_validation_scaler.shape

(730, 1)

In [38]:
X_test_scaler.shape

(547, 315)

In [39]:
y_test_scaler.shape

(547, 1)

### Create dataloader

In [40]:
train_dataset = TensorDataset(
    torch.tensor(X_train_scaler, dtype=torch.float32),
    torch.tensor(y_train_scaler, dtype=torch.float32),
)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [41]:
validation_dataset = TensorDataset(
    torch.tensor(X_validation_scaler, dtype=torch.float32),
    torch.tensor(y_validation_scaler, dtype=torch.float32),
)
validation_dataloader = DataLoader(validation_dataset, batch_size=32)

In [42]:
test_dataset = TensorDataset(
    torch.tensor(X_test_scaler, dtype=torch.float32),
    torch.tensor(y_test_scaler, dtype=torch.float32),
)
test_dataloader = DataLoader(test_dataset, batch_size=32)

### Create model

In [43]:
class LightningLSTM(L.LightningModule):

    def __init__(self):

        super().__init__()

        L.seed_everything(seed=18)

        self.lstm = nn.LSTM(input_size=INPUT_SIZE, hidden_size=64, batch_first=True)

        self.fc = nn.Linear(64, OUTPUT_SIZE)

    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)

        # Lấy hidden state cuối
        last_hidden = h_n[-1]

        out = self.fc(last_hidden)

        return out

    def configure_optimizers(
        self,
    ):  # this configures the optimizer we want to use for backpropagation.
        return Adam(
            self.parameters(), lr=LEARNING_RATE
        )  ## we'll just go ahead and set the learning rate to 0.1

    def training_step(self, batch):  # take a step during gradient descent.
        input_i, label_i = batch  # collect input
        output_i = self(input_i)  # run input through the neural network

        print(output_i.shape)
        print(label_i.shape)

        loss = F.mse_loss(output_i, label_i)

        self.log("train_loss", loss)

        return loss

    def validation_step(self, batch):  # take a step during validation.
        input_i, label_i = batch  # collect input
        output_i = self(input_i)  # run input through the neural network
        loss = F.mse_loss(output_i, label_i)

        self.log("val_loss", loss, prog_bar=True)

        return loss

In [44]:
model = LightningLSTM()

Seed set to 18


### Train model

In [45]:
early_stop_callback = EarlyStopping(
    monitor="val_loss",  # metric to watch
    patience=PATIENCE,  # epochs to wait before stopping
    mode="min",  # because lower loss is better
)

In [46]:
trainer = L.Trainer(
    max_epochs=1, log_every_n_steps=1, callbacks=[early_stop_callback]
)

trainer.fit(
    model, train_dataloaders=train_dataloader, val_dataloaders=validation_dataloader
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA GeForce RTX 3050 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name ┃ Type   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ lstm │ LSTM   │ 97.5 K │ train │     0 │
│ 1 │ fc   │ Linear │     65 │ train │     0 │
└───┴──────┴────────┴────────┴───────┴───────┘

Trainable params: 97.6 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 97.6 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 2                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

d:\GIT\master-thesis\mt_env\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.

...

torch.Size([1])

torch.Size([14, 1])

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_32352\2218374718.py:37: UserWarning: Using a target size 
(torch.Size([14, 1])) that is different to the input size (torch.Size([1])). This will likely lead to incorrect 
results due to broadcasting. Please ensure they have the same size.
  loss = F.mse_loss(output_i, label_i)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_32352\2218374718.py:46: UserWarning: Using a target size 
(torch.Size([26, 1])) that is different to the input size (torch.Size([1])). This will likely lead to incorrect 
results due to broadcasting. Please ensure they have the same size.
  loss = F.mse_loss(output_i, label_i)

`Trainer.fit` stopped: `max_epochs=1` reached.
